In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append(r'C:\Users\julia\OneDrive\Escritorio\Trabajo\building_ml_models_for_protein_science\src')


In [3]:
import pandas as pd
import os
import time
from building_models.utils.utils_functions import UtilsFunctions

In [4]:
path_data = "../../processed_dataset/antioxidant_classification/"
path_export = "../../processed_dataset/"

In [5]:
list_sources = os.listdir(path_data)
list_sources = [value for value in list_sources if value != "processed_dataset"]
list_sources

['Ahmad et al',
 'AMPDB v1',
 'ANOX',
 'AOD',
 'AOPxSVM',
 'Butt et al',
 'Feng et al',
 'Lam et al',
 'PredAoDP',
 'Thanh-Lam et al',
 'Zhai et al',
 'Zhang et al']

In [6]:
list_dfs = []

for source in list_sources:
    df = pd.read_csv(f"{path_data}/{source}/processed_data.csv")
    df["source"] = source
    list_dfs.append(df[["sequence", "label", "source"]])

df_all = pd.concat(list_dfs, ignore_index=True)

df_pivot = (
    df_all.pivot_table(
        index="sequence",
        columns="source",
        values="label",
        aggfunc="first"
    )
    .reset_index()
)

df_pivot.columns.name = None
df_pivot = df_pivot.fillna(999)

for column in df_pivot.columns:
    if column != "sequence":
        df_pivot[column] = df_pivot[column].astype(int)
print(df_pivot.shape)
df_pivot.head()

(7870, 13)


,sequence,AMPDB v1,ANOX,AOD,AOPxSVM,Ahmad et al,Butt et al,Feng et al,Lam et al,PredAoDP,Thanh-Lam et al,Zhai et al,Zhang et al
0,AAAAAAMTMMDMNFKYCHKIMKKHSKSFSYAFDLLPEDQRKAVWAI...,999,0,999,999,999,999,0,0,0,999,0,999
1,AAAAAGGGEGEG,999,999,999,1,999,999,999,999,999,999,999,999
2,AAAAG,999,999,999,1,999,999,999,999,999,999,999,999
3,AAAPVAVAK,999,999,999,1,999,999,999,999,999,999,999,999
4,AAASFGQTKIPRGNGPYSVGCTDLMFDHTNKGTFLRLYYPSQDNDR...,999,0,999,999,999,999,0,0,0,999,0,999


In [7]:
source_cols = [col for col in df_pivot.columns if col != "sequence"]

df_pivot["count_0"] = (df_pivot[source_cols] == 0).sum(axis=1)
df_pivot["count_1"] = (df_pivot[source_cols] == 1).sum(axis=1)

df_pivot["n_valid_sources"] = df_pivot[source_cols].isin([0, 1]).sum(axis=1)

df_pivot["count_0_norm"] = (
    df_pivot["count_0"] / df_pivot["n_valid_sources"].replace(0, pd.NA)
)

df_pivot["count_1_norm"] = (
    df_pivot["count_1"] / df_pivot["n_valid_sources"].replace(0, pd.NA)
)

df_pivot.head()

,sequence,AMPDB v1,ANOX,AOD,AOPxSVM,Ahmad et al,Butt et al,Feng et al,Lam et al,PredAoDP,Thanh-Lam et al,Zhai et al,Zhang et al,count_0,count_1,n_valid_sources,count_0_norm,count_1_norm
0,AAAAAAMTMMDMNFKYCHKIMKKHSKSFSYAFDLLPEDQRKAVWAI...,999,0,999,999,999,999,0,0,0,999,0,999,5,0,5,1.0,0.0
1,AAAAAGGGEGEG,999,999,999,1,999,999,999,999,999,999,999,999,0,1,1,0.0,1.0
2,AAAAG,999,999,999,1,999,999,999,999,999,999,999,999,0,1,1,0.0,1.0
3,AAAPVAVAK,999,999,999,1,999,999,999,999,999,999,999,999,0,1,1,0.0,1.0
4,AAASFGQTKIPRGNGPYSVGCTDLMFDHTNKGTFLRLYYPSQDNDR...,999,0,999,999,999,999,0,0,0,999,0,999,5,0,5,1.0,0.0


In [8]:
df_pivot_positive = df_pivot[df_pivot["count_1_norm"] == 1]
df_pivot_negative = df_pivot[df_pivot["count_0_norm"] == 1]

df_pivot_positive.shape, df_pivot_negative.shape

((2649, 18), (5178, 18))

In [9]:
df_pivot_positive["label"] = 1
df_pivot_negative["label"] = 0

In [10]:
df_processed = pd.concat([df_pivot_positive, df_pivot_negative], axis=0, ignore_index=True)
df_processed["label"].value_counts()

label
0    5178
1    2649
Name: count, dtype: int64

In [11]:
sequences_with_different_annotations = df_pivot[~df_pivot["sequence"].isin(df_processed["sequence"])]
print(sequences_with_different_annotations.shape)
sequences_with_different_annotations.head()

(43, 18)


,sequence,AMPDB v1,ANOX,AOD,AOPxSVM,Ahmad et al,Butt et al,Feng et al,Lam et al,PredAoDP,Thanh-Lam et al,Zhai et al,Zhang et al,count_0,count_1,n_valid_sources,count_0_norm,count_1_norm
2483,MAGQKIRIRLKAYDHEAIDASARKIVETVVRTGASVVGPVPLPTEK...,999,999,999,999,0,999,999,999,1,999,999,0,2,1,3,0.666667,0.333333
2509,MAHKKAGGSTRNGRDSESKRLGVKRFGGESVLAGNIIVRQRGTKFH...,999,999,999,999,0,999,999,999,1,999,999,0,2,1,3,0.666667,0.333333
2515,MAIDENKQKALAAALGQIEKQFGKGSIMRLGEDRSMDVETISTGSL...,999,999,999,999,0,999,999,999,1,999,999,0,2,1,3,0.666667,0.333333
2540,MAKISKRRQAFAAKVDRQKLYAIEDALSLVKECASAKFDESIDVAV...,999,999,999,999,0,999,999,999,1,999,999,0,2,1,3,0.666667,0.333333
2712,MARDIAAPPVPTNHQELISWVNEIAELTQPDAVVWCDGSEAEYERL...,999,999,999,999,0,999,999,999,1,999,999,0,2,1,3,0.666667,0.333333


- Exporting data

In [12]:
os.makedirs(f"{path_export}/merged_data", exist_ok=True)

In [13]:
df_processed["length"] = df_processed["sequence"].str.len()

In [14]:
df_processed.to_csv(f"{path_export}/merged_data/processed_dataset.csv", index=False)
sequences_with_different_annotations.to_csv(f"{path_export}/merged_data/sequences_with_erros.csv", index=False)

- Create metadata

In [16]:
dict_metadata = {
    "task" : "antioxidant_classification",
    "mode" : "binary",
    "number_of_examples": df_pivot.shape[0],
    "number_of_inconsistences" : sequences_with_different_annotations.shape[0],
    "number_unique_annotated" : df_processed.shape[0],
    "positive_examples" : df_pivot_positive.shape[0],
    "negative_examples" : df_pivot_negative.shape[0],
    "statistic_dataset": {
        "min_length": int(df_processed["length"].min()),
        "max_length" : int(df_processed["length"].max())
    },
    "date_process" : time.strftime("%Y-%m-%d %H:%M:%S")
}

dict_metadata

{'task': 'antioxidant_classification',
 'mode': 'binary',
 'number_of_examples': 7870,
 'number_of_inconsistences': 43,
 'number_unique_annotated': 7827,
 'positive_examples': 2649,
 'negative_examples': 5178,
 'statistic_dataset': {'min_length': 2, 'max_length': 8799},
 'date_process': '2026-04-08 16:51:38'}

In [17]:
UtilsFunctions.export_json(f"{path_export}/merged_data/metadata.json", dict_metadata)